# 04. Корреляции внутри страны (метод 12) и приложение M1 (метод 14)

## Метод 12
Для **одной** страны по годам считаем корреляцию Пирсона между двумя рядами. Это ассоциация совместного движения во времени, **не** причинно-следственная связь.

Пары для отчёта:
- GERD% ВВП и число статей;
- исследователи на млн и число статей;
- GERD% ВВП и TFP (`tfp_ctfp`).

## Метод 14 (только приложение)
Панельная регрессия с фиксированными эффектами страны и года (спецификация M1):  
`TFP_{it} = α_i + δ_t + β·GERD_{i,t−1} + θ·log(researchers_per_million)_{it} + ε_{it}`.

Почему не в сюжет: (а) у США в Penn World Table `ctfp = 1` каждый год **по построению**; (б) при кластерных стандартных ошибках доверительный интервал для β_GERD включает ноль. Честная фраза: «в этой спецификации устойчивой связи нет».


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = Path("..").resolve()
PANEL_PATH = ROOT / "data_reviewed" / "core_panel_reviewed.csv"
TABLES = ROOT / "data_reviewed" / "tables_reviewed"
RAW = ROOT / "data" / "raw"
OUT_FIG = ROOT / "notebooks" / "figures"
OUT_TAB = ROOT / "notebooks" / "tables"
OUT_FIG.mkdir(parents=True, exist_ok=True)
OUT_TAB.mkdir(parents=True, exist_ok=True)

panel = pd.read_csv(PANEL_PATH)
panel = panel[(panel["year"] >= 2010) & (panel["year"] <= 2023)].copy()
US_CN = panel[panel["country_iso3"].isin(["USA", "CHN"])].copy()

def cagr(start: float, end: float, n_years: int) -> float:
    """Compound annual growth rate; n_years = end_year - start_year."""
    if start is None or end is None or np.isnan(start) or np.isnan(end) or start <= 0 or n_years <= 0:
        return np.nan
    return (end / start) ** (1 / n_years) - 1

print("panel rows", len(panel), "US+CN rows", len(US_CN))
print("countries", sorted(panel["country_iso3"].unique()))


panel rows 112 US+CN rows 28
countries ['CHN', 'DEU', 'FRA', 'GBR', 'ISR', 'JPN', 'KOR', 'USA']


In [2]:
pairs = [
    ("gerd_pct_gdp", "scopus_articles"),
    ("researchers_per_million", "scopus_articles"),
    ("gerd_pct_gdp", "tfp_ctfp"),
]
rows = []
for iso in ["USA", "CHN"]:
    sub = panel[panel["country_iso3"] == iso]
    for x, y in pairs:
        xy = sub[["year", x, y]].dropna()
        n = len(xy)
        corr = xy[x].corr(xy[y]) if n >= 5 else np.nan
        rows.append({
            "country": iso,
            "x": x,
            "y": y,
            "pearson_r": corr,
            "n_years": n,
            "year_min": int(xy["year"].min()) if n else np.nan,
            "year_max": int(xy["year"].max()) if n else np.nan,
        })
corr_tbl = pd.DataFrame(rows)
corr_tbl.to_csv(OUT_TAB / "04_within_country_correlations.csv", index=False)
corr_tbl


C:\Users\danii\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\numpy\lib\_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\danii\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\numpy\lib\_function_base_impl.py:3066: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


,country,x,y,pearson_r,n_years,year_min,year_max
0,USA,gerd_pct_gdp,scopus_articles,0.775636,14,2010,2023
1,USA,researchers_per_million,scopus_articles,0.912085,13,2010,2022
2,USA,gerd_pct_gdp,tfp_ctfp,NaN,14,2010,2023
3,CHN,gerd_pct_gdp,scopus_articles,0.965279,14,2010,2023
4,CHN,researchers_per_million,scopus_articles,0.991613,14,2010,2023
5,CHN,gerd_pct_gdp,tfp_ctfp,0.954060,14,2010,2023


In [3]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=False)
for ax, iso in zip(axes, ["USA", "CHN"]):
    sub = panel[panel["country_iso3"] == iso][["year", "gerd_pct_gdp", "scopus_articles"]].dropna()
    ax.scatter(sub["gerd_pct_gdp"], sub["scopus_articles"], c=sub["year"], cmap="viridis")
    ax.set_title(f"{iso}: GERD% vs статьи")
    ax.set_xlabel("GERD, % ВВП")
    ax.set_ylabel("Статьи Scopus")
    ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_FIG / "04_within_corr_gerd_articles.png", dpi=150)
plt.show()


C:\Users\danii\AppData\Local\Temp\ipykernel_22232\1037418598.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Приложение: M1 (не пересчитываем, читаем канон)

Числа из `regression_results_final.csv` / `M1_INTERPRETATION.md`. Не подбираем лаги ради значимости.


In [4]:
m1_path = ROOT / "regression_results_final.csv"
m1 = pd.read_csv(m1_path)
# Keep a compact appendix view if columns exist; otherwise show head.
m1.to_csv(OUT_TAB / "04_m1_appendix_source.csv", index=False)
print("M1 source rows:", len(m1))
m1.head(20)


M1 source rows: 9


,model,term,beta,se_hc1,p_hc1,ci_lo_hc1,ci_hi_hc1,se_cluster_country,p_cluster_Gminus1,ci_lo_cluster95_tGminus1,ci_hi_cluster95_tGminus1,N,K,df_resid,G,status,note
0,M1_TFP_FE_retained,gerd_lag1,0.02746,0.01427,0.059,-0.00051,0.05543,0.03100,0.409,-0.04833,0.10325,84,20,64,7,RETAINED_AS_DESCRIPTIVE_ONLY,"HC1 CI includes 0; cluster-robust (G=7, t_6 cr..."
1,M1_TFP_FE_retained,lres_logResearchers,0.05629,0.00429,0.000,0.04787,0.06470,0.00792,0.000,0.03690,0.07567,84,20,64,7,RETAINED_AS_DESCRIPTIVE_ONLY,HC1 t=13.1 inflated by serial correlation; clu...
2,M1_sensitivity_lag0_contemp,gerd_contemp,0.01250,0.01320,0.418,NaN,NaN,NaN,NaN,NaN,NaN,91,21,70,7,SENSITIVITY,Contemporaneous GERD insignificant; shows lag-...
3,M1_sensitivity_lag2,gerd_lag2,0.05250,0.01580,0.001,NaN,NaN,NaN,NaN,NaN,NaN,77,20,57,7,SENSITIVITY,Lag-2 doubles beta vs lag-1; lag structure min...
4,M1_sensitivity_dropUSA,gerd_lag1,0.03290,0.01540,0.038,NaN,NaN,NaN,0.368,NaN,NaN,72,19,53,6,SENSITIVITY,Dropping USA (TFP-constant anchor) raises beta...
5,M1_sensitivity_dropCHN,gerd_lag1,0.01890,0.01320,0.157,NaN,NaN,NaN,0.507,NaN,NaN,71,19,52,6,SENSITIVITY,Dropping CHN halves beta and loses significanc...
6,M1_sensitivity_dropKOR,gerd_lag1,0.04870,0.01990,0.017,NaN,NaN,NaN,0.353,NaN,NaN,71,19,52,6,SENSITIVITY,Dropping high-GERD KOR nearly doubles beta; si...
7,M1_sensitivity_2011_2021,gerd_lag1,0.03530,0.01530,0.024,NaN,NaN,NaN,0.301,NaN,NaN,66,17,49,7,SENSITIVITY,Balanced-window beta larger than baseline; win...
8,M2_TFP_PC1,pc1_lag1,-0.11796,0.00686,0.000,NaN,NaN,NaN,NaN,NaN,NaN,91,20,71,7,EXCLUDED_DO_NOT_USE,"Wrong sign (more tech -> lower TFP), PCA loadi..."


### Обязательная оговорка по TFP
В Penn World Table показатель `ctfp` для США равен 1 **каждый год по построению** (Feenstra, Inklaar, Timmer, 2015). Поэтому within-оценка для США по TFP не информативна как «рост производительности США».

### Разрешённый язык
«условная ассоциация», «неотличимо от нуля под cluster SE», «дескриптивно / приложение».  
**Запрещено:** «эффект GERD на TFP», «доказательство конверсии».
